In [22]:
import sys
import os
from pathlib import Path

project_root = Path("/content/drive/MyDrive/CCTV_VLM_Project")

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

os.chdir(project_root)

In [23]:
import cv2
import torch
import numpy as np
import supervision as sv
import pandas as pd

from trackers import ByteTrackTracker

from src.config import (
    MODELS_DIR,
    VIDEOS_DIR,
    CLASS_MAPPING,
    DINO_PROMPT,
    YOLO_CONF_THRESH,
    DINO_BOX_THRESH,
    DINO_TEXT_THRESH,
    NMS_IOU_THRESH
)

from src.yolo_detector import YoloDetector
from src.dino_detector import DinoDetector
from src.fusion import BoxFusion

print("Supervision:", sv.__version__)
print("CUDA:", torch.cuda.is_available())

Supervision: 0.31.5
CUDA: True


In [24]:
video_path = VIDEOS_DIR / "warehouse2.mp4"
yolo_weight_path = MODELS_DIR/ "yolo11n_E3_PPE_best.pt"

yolo = YoloDetector(
    model_path = str(yolo_weight_path),
    conf = YOLO_CONF_THRESH
)

dino = DinoDetector(
    box_thresh = DINO_BOX_THRESH,
    text_thresh = DINO_TEXT_THRESH
)

fusion = BoxFusion(
    iou_threshold = NMS_IOU_THRESH
)

print("All models initialized")

Loading weights:   0%|          | 0/978 [00:00<?, ?it/s]

All models initialized


In [25]:
tracker = ByteTrackTracker(
    track_activation_threshold=0.25,
    lost_track_buffer = 30,
    frame_rate = 28,
    minimum_iou_threshold = 0.5,
    minimum_consecutive_frames=1,
    high_conf_det_threshold = 0.25
)

print("Tracker initialized.")

Tracker initialized.


In [26]:
cap = cv2.VideoCapture(str(video_path))
ret , frame = cap.read()

cap.release()

assert ret , "Could not read video frame"

print(frame.shape)

(1080, 1920, 3)


In [27]:
yolo_det = yolo.detect(frame)

dino_det = dino.detect(
    frame , 
    prompt = DINO_PROMPT
)

fused_det = fusion.merge_and_suppress(
    yolo_det,
    dino_Det
)
print("YOLO detections:", len(yolo_det))
print("DINO detections:", len(dino_det))
print("Fused detections:", len(fused_det))

print("\nFused class distribution:")

for class_id, class_name in CLASS_MAPPING.items():
    count = np.sum(fused_det.class_id == class_id)
    print(f"{class_name}: {count}")


YOLO detections: 2
DINO detections: 8
Fused detections: 8

Fused class distribution:
person: 6
forklift: 0
pallet_truck: 2


In [28]:
tracked_det = tracker.update(
    detections = fused_det
)

print("Input fused detections:", len(fused_det))
print("Tracker output detections:", len(tracked_det))

print("\nTracker IDs:")
print(tracked_det.tracker_id)

print("\nTracked class IDs:")
print(tracked_det.class_id)

Input fused detections: 8
Tracker output detections: 8

Tracker IDs:
[-1 -1 -1 -1 -1 -1 -1 -1]

Tracked class IDs:
[0 0 0 0 2 2 0 0]


In [29]:
print("Fused boxes:", len(fused_det))
print("Returned boxes:", len(tracked_det))

print("Assigned IDs:", np.sum(tracked_det.tracker_id >= 0))
print("Unassigned IDs:", np.sum(tracked_det.tracker_id < 0))

print("\nClass-wise tracking:")

for class_id, class_name in CLASS_MAPPING.items():
    mask = tracked_det.class_id == class_id

    class_tracks = tracked_det.tracker_id[mask]

    print(
        f"{class_name}: "
        f"boxes={len(class_tracks)}, "
        f"assigned={np.sum(class_tracks >= 0)}, "
        f"IDs={class_tracks.tolist()}"
    )

Fused boxes: 8
Returned boxes: 8
Assigned IDs: 0
Unassigned IDs: 8

Class-wise tracking:
person: boxes=6, assigned=0, IDs=[-1, -1, -1, -1, -1, -1]
forklift: boxes=0, assigned=0, IDs=[]
pallet_truck: boxes=2, assigned=0, IDs=[-1, -1]


In [30]:
for i in range(len(fused_det)):
    print(
        f"Detections{i}:"
        f"Class = {CLASS_MAPPING[int(fused_det.class_id[i])]},"
        f"Confidence = {fused_det.confidence[i]:.4f},"
        f"Box = {fused_det.xyxy[i]}"
    )

Detections0:Class = person,Confidence = 0.7857,Box = [     691.78      857.96      798.54      1075.9]
Detections1:Class = person,Confidence = 0.3919,Box = [     406.72      303.52      454.88      403.52]
Detections2:Class = person,Confidence = 0.6253,Box = [     1466.9       407.1      1520.4       481.7]
Detections3:Class = person,Confidence = 0.5565,Box = [      476.1      458.37      531.02      531.56]
Detections4:Class = pallet_truck,Confidence = 0.4716,Box = [       1310      468.58        1598      800.94]
Detections5:Class = pallet_truck,Confidence = 0.4264,Box = [     135.11      467.39      709.06      1076.4]
Detections6:Class = person,Confidence = 0.3860,Box = [     1599.2      448.47      1641.3      534.62]
Detections7:Class = person,Confidence = 0.3529,Box = [     342.98      411.78      391.17      499.16]


In [31]:
test_tracker = ByteTrackTracker(
    track_activation_threshold=0.25,
    minimum_consecutive_frames=1
)

for i in range(5):
    result = test_tracker.update(detections=fused_det)

    print(
        f"Iteration {i}: "
        f"IDs={result.tracker_id.tolist()}"
    )

Iteration 0: IDs=[-1, -1, -1, -1, -1, -1, -1, -1]
Iteration 1: IDs=[0, 1, -1, -1, -1, -1, -1, -1]
Iteration 2: IDs=[0, 1, -1, -1, -1, -1, -1, -1]
Iteration 3: IDs=[0, 1, -1, -1, -1, -1, -1, -1]
Iteration 4: IDs=[0, 1, -1, -1, -1, -1, -1, -1]


In [15]:
print("Internal tracks:", len(test_tracker.tracks))

for track in test_tracker.tracks:
    print(vars(track))

Internal tracks: 2
{'age': 4, 'state_estimator': <trackers.utils.state_representations.XYXYStateEstimator object at 0x7dd05ca36d50>, 'tracker_id': 0, 'time_since_update': 0, 'time_since_update_seconds': 0.0, 'number_of_successful_consecutive_updates': 5}
{'age': 4, 'state_estimator': <trackers.utils.state_representations.XYXYStateEstimator object at 0x7dd05cbb0a50>, 'tracker_id': 1, 'time_since_update': 0, 'time_since_update_seconds': 0.0, 'number_of_successful_consecutive_updates': 5}


In [16]:
import inspect
print(inspect.signature(ByteTrackTracker))

(lost_track_buffer: int = 30, frame_rate: float = 30.0, track_activation_threshold: float = 0.7, minimum_consecutive_frames: int = 2, minimum_iou_threshold: float = 0.1, high_conf_det_threshold: float = 0.6, state_estimator_class: type[trackers.utils.state_representations.BaseStateEstimator] = <class 'trackers.utils.state_representations.XYXYStateEstimator'>, iou: trackers.utils.iou.BaseIoU | None = None) -> None


In [ ]:
|